In [131]:
import pandas as pd

In [132]:
excel_file = pd.ExcelFile(
    r"C:\Users\HP\Documents\Data Analysis\Python Analytics\Agent Transport.xlsx"
)

In [133]:
agent_trans_df = pd.read_excel(excel_file)

In [134]:
agent_trans_df = agent_trans_df.drop(columns=[
    "Last Login",
    "Pre Mid-night mins_",
    "Post Mid-night mins_",
    "Unnamed: 15",
    "Unnamed: 17"
])

In [135]:
# Coming from excel, some actual login rows are in string, so we convert the to time delta to enable proper calculation

agent_trans_df["Actual Login"] = pd.to_timedelta(
    agent_trans_df["Actual Login"].apply(
        lambda x: x.strftime("%H:%M:%S") if hasattr(x, "strftime") else str(x)
    )
)

In [136]:
# Function for  calculating the agaynt "Payout"

def calculate_payout(row):

    shift = row["Shift_R1"]
    att = row["Attendance status"]
    login_time = pd.to_timedelta(row["Actual Login"], errors="coerce")

    if pd.isna(login_time):
        return 0

    if att != "P":
        return 0

    if shift in ["A1", "AM2"] and login_time > pd.Timedelta("6:30:00"):
        return 3000

    elif shift == "MD" and login_time >= pd.Timedelta("7:00:00"):
        return 2000

    elif shift == "PTE2" and login_time >= pd.Timedelta("4:30:00"):
        return 3000

    elif shift == "N" and login_time >= pd.Timedelta("1:00:00"):
        return 3000

    else:
        return 0


In [137]:
# Inserrt the Pyaout Column and call function

agent_trans_df.insert(
    len(agent_trans_df.columns),
    "Payout",
    agent_trans_df.apply(calculate_payout, axis=1)
)

In [138]:
# Remove time (00:00:00) from date

agent_trans_df["APR Date_2"] = pd.to_datetime(
    agent_trans_df["APR Date_2"]
).dt.date

In [139]:
# Remove days from Actual Login Output

agent_trans_df["Actual Login"] = (
    agent_trans_df["Actual Login"]
    .astype(str)
    .str.replace("0 days ", "", regex=False)
)

In [130]:
agent_trans_df.to_excel("Payout Agent Transport.xlsx", index=False)